# Gold table creation

In [0]:
df_facilities = spark.sql(f"select * from regis_healthcare.silver.facilities;")
df_facilities.createOrReplaceTempView("facilities")

df_residents = spark.sql(f"select * from regis_healthcare.silver.residents;")
df_residents.createOrReplaceTempView("residents")

df_employees = spark.sql(f"select * from regis_healthcare.silver.employees;")
df_employees.createOrReplaceTempView("employees")

df_admissions = spark.sql(f"select * from regis_healthcare.silver.admissions;")
df_admissions.createOrReplaceTempView("admissions")

df_discharges = spark.sql(f"select * from regis_healthcare.silver.discharges;")
df_discharges.createOrReplaceTempView("discharges")

df_medications = spark.sql(f"select * from regis_healthcare.silver.medications;")
df_medications.createOrReplaceTempView("medications")

df_incidents = spark.sql(f"select * from regis_healthcare.silver.incidents;")
df_incidents.createOrReplaceTempView("incidents")

df_appointments = spark.sql(f"select * from regis_healthcare.silver.appointments;")
df_appointments.createOrReplaceTempView("appointments")

df_billing = spark.sql(f"select * from regis_healthcare.silver.billing;")
df_billing.createOrReplaceTempView("billing")

df_resident_feedback = spark.sql(f"select * from regis_healthcare.silver.resident_feedback;")
df_resident_feedback.createOrReplaceTempView("resident_feedback")

In [0]:
print(f"facilities = {df_facilities.columns}")
print(f"residents = {df_residents.columns}")
print(f"employees = {df_employees.columns}")
print(f"admissions = {df_admissions.columns}")
print(f"discharges = {df_discharges.columns}")
print(f"medications = {df_medications.columns}")
print(f"incidents = {df_incidents.columns}")
print(f"appointments= {df_appointments.columns}")
print(f"billing = {df_billing.columns}")
print(f"resident_feedback = {df_resident_feedback.columns}")

#### Dimenction Table Creation

##### 1.Dim_facilities

In [0]:
# dim_facilities --> Source: facilities

dim_facilities = spark.sql("""select * from facilities order by facility_id""")
# display(dim_facilities)
# #--------------------------
# from pyspark.sql.functions import col, date_format
# # Create date_key column in YYYYMMDD format
# dim_facilities = dim_facilities.withColumn("cr_at_date_key", date_format(col("created_at"), "yyyyMMdd"))
# # Optionally cast to integer for warehouse-style keys
# dim_facilities = dim_facilities.withColumn("cr_at_date_key", col("cr_at_date_key").cast("int"))
# #--------------------------
from pyspark.sql.functions import col, regexp_replace
dim_facilities = dim_facilities.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(dim_facilities)
dim_facilities = dim_facilities.select(
    "facility_key",
    "facility_id",
    "facility_name",
    "address",
    "suburb",
    "state",
    "postcode",
    "phone",
    "email",
    "capacity",
    "accreditation_status",
    "created_at")
display(dim_facilities)

In [0]:
# # cataloge 
# dim_facilities.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_facilities")

In [0]:
dim_facilities.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_facilities")
print(dim_facilities.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_facilities")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_facilities")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.facility_key = source.facility_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_facilities;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_facilities;")
print(sb_dim_df.count())

In [0]:
# # gold load to s3
# dim_facilities.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_facilities")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_facilities"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = dim_facilities

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.facility_key = source.facility_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)


##### 2. Dim_Resident

In [0]:
# 2. Dim_Resident --> Source: residents

Dim_Resident = spark.sql("""select * from residents order by resident_id""")
# display(Dim_Resident)
# #------------------
# from pyspark.sql.functions import col, date_format

# # Create date_key column in YYYYMMDD format
# df_residents = df_residents.withColumn("dob_date_key", date_format(col("date_of_birth"), "yyyyMMdd"))
# # Optionally cast to integer for warehouse-style keys
# df_residents = df_residents.withColumn("dob_date_key", col("dob_date_key").cast("int"))
# #------------------

from pyspark.sql.functions import col, regexp_replace
Dim_Resident = Dim_Resident.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
# display(Dim_Resident)
Dim_Resident = Dim_Resident.select(
    "resident_key",
    "resident_id",
    "first_name",
    "last_name",
    "date_of_birth",
    "gender",
    "address",
    "suburb",
    "state",
    "postcode",
    "phone",
    "email",
    "care_level",
    "medicare_number"
)
display(Dim_Resident)

In [0]:
# # cataloge 
# Dim_Resident.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_Resident")

In [0]:
Dim_Resident.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_Resident")
print(Dim_Resident.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_Resident")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_Resident")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.resident_key = source.resident_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_Resident;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_Resident;")
print(sb_dim_df.count())

In [0]:
# gold load to s3
Dim_Resident.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/dim_Resident")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_Resident"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = Dim_Resident

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.resident_key = source.resident_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)


##### 3. Dim_Employee

In [0]:
# 3. Dim_Employee --> Source: employees
# | Column          |
# | --------------- |
# | employee_key    |
# | employee_id     |
# | first_name      |
# | last_name       |
# | job_title       |
# | employment_type |
# | status          |
# | salary          |
# | hire_date       |


Dim_Employee = spark.sql("""select * from employees order by employee_id""")
# display(Dim_Employee)

from pyspark.sql.functions import col, regexp_replace
Dim_Employee = Dim_Employee.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)
# from pyspark.sql.functions import col, date_format
# # Create date_key column in YYYYMMDD format
# Dim_Employee = Dim_Employee.withColumn("emp_hire_date_key", date_format(col("hire_date"), "yyyyMMdd"))
# # Optionally cast to integer for warehouse-style keys
# Dim_Employee = Dim_Employee.withColumn("emp_hire_date_key", col("emp_hire_date_key").cast("int"))
# display(df_employees)
Dim_Employee = Dim_Employee.select(
    "employee_key",
    "employee_id",
    "first_name",
    "last_name",
    "job_title",
    "employment_type",
    "status",
    "salary",
    "hire_date")
display(Dim_Employee)

In [0]:
# 4. Dim_Date -- > Create from all date columns.
# | Column              |
# | ------------------- |
# | date_key (YYYYMMDD) |
# | full_date           |
# | day                 |
# | month               |
# | month_name          |
# | quarter             |
# | year                |
# | week_no             |
# | day_name            |

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

date_df = spark.sql("""
SELECT explode(
    sequence(
        to_date('2020-01-01'),
        to_date('2035-12-31'),
        interval 1 day
    )
) AS full_date
""")

dim_date = (
    date_df
    .withColumn("date_key",
                date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("quarter", concat(lit("Q"), quarter("full_date")))
    .withColumn("year", year("full_date"))
    .withColumn("week_no", weekofyear("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn(
        "is_weekend",
        when(dayofweek("full_date").isin(1,7), "Y").otherwise("N")
    )
)

display(dim_date)

In [0]:
# 5. Dim_Medication -- >Source: medications
# | Column             |
# | ------------------ |
# | medication_key     |
# | medication_id      |
# | medication_name    |
# | dosage             |
# | frequency          |
# | prescribing_doctor |
Dim_Medication = spark.sql("""select * from medications order by medication_id""")
# display(Dim_Medication)
from pyspark.sql.functions import col, regexp_replace
Dim_Medication = Dim_Medication.withColumn(
    "medication_key",
    regexp_replace(col("medication_id"), "^MED", "").cast("int")
)
# display(df_medicationss)
Dim_Medication = Dim_Medication.select(
  "medication_key",     
  "medication_id",      
  "medication_name",    
  "dosage",             
  "frequency",         
  "prescribing_doctor" 
)
display(Dim_Medication)

In [0]:
# 6. Dim_Incident_Type -- >Source: incidents
# | Column            |
# | ----------------- |
# | incident_type_key |
# | incident_type     |
# | severity          |

from pyspark.sql.functions import col,when
Dim_Incident_Type = df_incidents.withColumn("incident_type_key",when(col("incident_type")=="AGGRESSION",1)
     .when(col("incident_type")=="BEHAVIOURAL ISSUE",2)
     .when(col("incident_type")=="CHOKING",3)
     .when(col("incident_type")=="ELOPEMENT",4)
     .when(col("incident_type")=="EQUIPMENT FAILURE",5)
     .when(col("incident_type")=="FALL",6)
     .when(col("incident_type")=="INFECTION",7)
     .when(col("incident_type")=="MEDICATION ERROR",8)
     .when(col("incident_type")=="OTHER",9)
     .when(col("incident_type")=="PRESSURE INJURY",10)
     .when(col("incident_type")=="SKIN TEAR",11)
     .when(col("incident_type")=="UNKNOWN",12)
     .when(col("incident_type")=="WANDERING",13).otherwise(0).cast("int"))


Dim_Incident_Type = Dim_Incident_Type.select(
     "incident_type_key",
    "incident_type",
    "severity" 
)
display(Dim_Incident_Type)


In [0]:
# 7. Dim_Billing_Type
# | Column           |
# | ---------------- |
# | billing_type_key |
# | billing_type     |
# display(df_billing)
#----------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

Dim_Billing = spark.sql("""
    select distinct(billing_type) from billing""")

Dim_Billing.createOrReplaceTempView("bill")

Dim_Billing = spark.sql("""
    select billing_type,row_number()over(order by billing_type )as billing_type_key from bill group by billing_type """)
#----------------
Dim_Billing = Dim_Billing.select(
"billing_type_key",
"billing_type")
display(Dim_Billing)


In [0]:
# 8. Dim_Appointment
# | Column               |
# | -------------------- |
# | appointment_type_key |
# | appointment_type     |

Dim_Appointment = spark.sql("""select * from appointments""")
# display(Dim_Appointment)
#----------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Dim_Appointment = Dim_Appointment.select(
#     "appointment_type"
# ).distinct()
# window_spec = Window.orderBy("appointment_type")
Dim_Appointment = spark.sql("""
    select distinct(appointment_type) from appointments """)
Dim_Appointment.createOrReplaceTempView("aps")
# Dim_Appointment = Dim_Appointment.withColumn(
#     "appointment_type_key",
#     row_number().over(window_spec)
# )
Dim_Appointment = spark.sql("""
    select appointment_type,row_number()over(order by appointment_type )as appointment_type_key from aps group by appointment_type """)
#----------------
Dim_Appointment = Dim_Appointment.select(
"appointment_type_key",
"appointment_type")
display(Dim_Appointment)

#### Fact Table Creation

In [0]:
# Fact_Admissions --> Source: admissions

# | Foreign Keys       |
# | ------------------ |
# | admission_key      |
# | resident_key       |
# | facility_key       |
# | admission_date_key |

#--------------------------

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
df_admissions = df_admissions.withColumn("admission_date_key", date_format(col("admission_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
df_admissions = df_admissions.withColumn("admission_date_key", col("admission_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
df_admissions = df_admissions.withColumn(
    "admission_key",
    regexp_replace(col("admission_id"), "^ADM", "").cast("int")
)
df_admissions = df_admissions.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
df_admissions = df_admissions.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(df_facilities)
df_admissions = df_admissions.select(
 "admission_key",   
 "resident_key",       
 "facility_key",       
 "admission_date_key"
)
display(df_admissions)



In [0]:
# Fact_Discharges -- > Source: discharges
# | Foreign Keys       |
# | ------------------ |
# | discharge_key      |
# | resident_key       |
# | facility_key       |
# | discharge_date_key |

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
df_discharges = df_discharges.withColumn("discharge_date_key", date_format(col("discharge_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
df_discharges = df_discharges.withColumn("discharge_date_key", col("discharge_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
df_discharges = df_discharges.withColumn(
    "discharge_key",
    regexp_replace(col("discharge_id"), "^DIS", "").cast("int")
)
df_discharges = df_discharges.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
df_discharges = df_discharges.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(df_facilities)
df_discharges = df_discharges.select(
 "discharge_key",      
 "resident_key",       
 "facility_key",       
 "discharge_date_key" 
)
display(df_discharges)


In [0]:
# Fact_Medications --> Source: medications
# | Foreign Keys        |
# | ------------------- |
# | medication_fact_key |
# | resident_key        |
# | medication_key      |
# | start_date_key      |
# | end_date_key        |
# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number

# window_spec = Window.orderBy("medication_id")

# df_medications = df_medications.withColumn(
#     "medication_fact_key",
#     row_number().over(window_spec)
# )
df_medications = spark.sql("""SELECT 
    *,
    ROW_NUMBER() OVER (ORDER BY medication_id) AS medication_fact_key
FROM medications;
""")
#---- existing table id
# from pyspark.sql.functions import max

# max_key = df_medications.agg(
#     max("medication_fact_key")
# ).collect()[0][0]

# max_key = max_key if max_key else 0

# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number

# window_spec = Window.orderBy("medication_id")

# df_medications = df_medications.withColumn(
#     "medication_fact_key",
#     row_number().over(window_spec) + max_key
# )
# display(df_medications)
#-------------------------------
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
df_medications = df_medications.withColumn("start_date_key", date_format(col("start_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
df_medications = df_medications.withColumn("start_date_key", col("start_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
df_medications = df_medications.withColumn("end_date_key", date_format(col("end_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
df_medications = df_medications.withColumn("end_date_key", col("end_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
df_medications = df_medications.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
df_medications = df_medications.withColumn(
    "medication_key",
    regexp_replace(col("medication_id"), "^MED", "").cast("int")
)
# display(df_facilities)
df_medications = df_medications.select(
 "medication_fact_key", 
 "resident_key",        
 "medication_key",      
 "start_date_key",     
 "end_date_key"   
)
display(df_medications)



In [0]:
# Fact_Incidents -- > Source: incidents
# | Foreign Keys      |
# | ----------------- |
# | incident_key      |
# | resident_key      |
# | facility_key      |
# | employee_key      |
# | incident_type_key |
# | incident_date_key |

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
Fact_Incidents = df_incidents.withColumn("incident_date_key", date_format(col("incident_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
Fact_Incidents = Fact_Incidents.withColumn("incident_date_key", col("incident_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
Fact_Incidents = Fact_Incidents.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
Fact_Incidents = Fact_Incidents.withColumn(
    "incident_key",
    regexp_replace(col("incident_id"), "^INC", "").cast("int")
)

Fact_Incidents = Fact_Incidents.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)

Fact_Incidents = Fact_Incidents.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)
from pyspark.sql.functions import col,when

Fact_Incidents = Fact_Incidents.withColumn("incident_type_key",when(col("incident_type")=="AGGRESSION",1)
     .when(col("incident_type")=="BEHAVIOURAL ISSUE",2)
     .when(col("incident_type")=="CHOKING",3)
     .when(col("incident_type")=="ELOPEMENT",4)
     .when(col("incident_type")=="EQUIPMENT FAILURE",5)
     .when(col("incident_type")=="FALL",6)
     .when(col("incident_type")=="INFECTION",7)
     .when(col("incident_type")=="MEDICATION ERROR",8)
     .when(col("incident_type")=="OTHER",9)
     .when(col("incident_type")=="PRESSURE INJURY",10)
     .when(col("incident_type")=="SKIN TEAR",11)
     .when(col("incident_type")=="UNKNOWN",12)
     .when(col("incident_type")=="WANDERING",13).otherwise(0).cast("int"))

Fact_Incidents = Fact_Incidents.select(
 "incident_key",     
 "resident_key",    
 "facility_key",   
 "employee_key", 
 "incident_type_key", 
 "incident_date_key" 
)

display(Fact_Incidents)


In [0]:
# Fact_Appointments -- > Source: appointments
# | Foreign Keys         |
# | -------------------- |
# | appointment_key      |
# | resident_key         |
# | employee_key         |
# | appointment_type_key |
# | appointment_date_key |
# --$
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_app = df_appointments.select(
    "appointment_type"
).distinct()
window_spec = Window.orderBy("appointment_type")

df_app = df_app.withColumn(
    "appointment_type_key",
    row_number().over(window_spec)
)
df_app = df_app.select(
"appointment_type_key",
"appointment_type")
df_app.createOrReplaceTempView("appointmentss")
Fact_Appointments =spark.sql("""
 select a.*,s.appointment_type_key from appointments as a join appointmentss as s 
 on a.appointment_type = s.appointment_type
""")
# #--$
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
Fact_Appointments = Fact_Appointments.withColumn("appointment_date_key", date_format(col("appointment_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
Fact_Appointments = Fact_Appointments.withColumn("appointment_date_key", col("appointment_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
Fact_Appointments = Fact_Appointments.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
Fact_Appointments = Fact_Appointments.withColumn(
    "appointment_key",
    regexp_replace(col("appointment_id"), "^APT", "").cast("int")
)

Fact_Appointments = Fact_Appointments.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)

Fact_Appointments = Fact_Appointments.select(
"appointment_key",     
 "resident_key",      
 "employee_key",     
 "appointment_type_key", 
 "appointment_date_key" 
)

display(Fact_Appointments)



In [0]:
# Fact_Billing -- > Source: billing
# | Foreign Keys     |
# | ---------------- |
# | billing_key      |
# | resident_key     |
# | facility_key     |
# | billing_type_key |
# | billing_date_key |
# | payment_date_key |

# display(df_billing)
# display(Dim_Billing)
from pyspark.sql.functions import col, date_format
#-------------

Dim_Billing.createOrReplaceTempView("dim")

Fact_Billing = spark.sql("""
select t.*,d.billing_type_key from dim as d join  billing as t on d.billing_type = t.billing_type""")
#------------
# Create date_key column in YYYYMMDD format
Fact_Billing = Fact_Billing.withColumn("payment_date_key", date_format(col("payment_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
Fact_Billing = Fact_Billing.withColumn("payment_date_key", col("payment_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
Fact_Billing = Fact_Billing.withColumn(
    "billing_key",
    regexp_replace(col("billing_id"), "^BIL", "").cast("int")
)
Fact_Billing = Fact_Billing.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)

Fact_Billing = Fact_Billing.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# Create date_key column in YYYYMMDD format
Fact_Billing = Fact_Billing.withColumn("billing_date_key", date_format(col("billing_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
Fact_Billing = Fact_Billing.withColumn("billing_date_key", col("billing_date_key").cast("int"))
# #-------------------------
# display(Fact_Billing)
Fact_Billing = Fact_Billing.select(
 "billing_key" ,     
 "resident_key" ,    
 "facility_key"  ,   
 "billing_type_key", 
 "billing_date_key" ,
 "payment_date_key" 
)

display(Fact_Billing)



In [0]:
# Fact_Resident_Feedback -- > Source: resident_feedback
# | Foreign Keys      |
# | ----------------- |
# | feedback_key      |
# | resident_key      |
# | facility_key      |
# | feedback_date_key |
Fact_Resident_Feedback = df_resident_feedback.withColumn(
    "feedback_key",
    regexp_replace(col("feedback_id"), "^FBK", "").cast("int")
)
Fact_Resident_Feedback = Fact_Resident_Feedback.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
Fact_Resident_Feedback = Fact_Resident_Feedback.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
Fact_Resident_Feedback = Fact_Resident_Feedback.withColumn("feedback_date_key", date_format(col("feedback_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
Fact_Resident_Feedback = Fact_Resident_Feedback.withColumn("feedback_date_key", col("feedback_date_key").cast("int"))
#-------------------------
Fact_Resident_Feedback = Fact_Resident_Feedback.select(
 "feedback_key",      
 "resident_key",      
 "facility_key",      
 "feedback_date_key" )
display(Fact_Resident_Feedback)
